# 06 — Physics-Informed Neural Network (PINN)

Loss = MSE_data + lambda * MSE(Q_pred, anchor_physics(OSNR)).

In [1]:
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))
import numpy as np, torch, mlflow, joblib
from torch.utils.data import DataLoader, TensorDataset
from src.data import load_processed, load_scaler
from src.models.pinn import PINNRegressor, pinn_loss
from src.evaluate import regression_metrics
from src.utils import set_seed
from src.config import MLFLOW_URI, EXPERIMENT_NAME, MODELS_DIR, FEATURE_COLS
set_seed(42); mlflow.set_tracking_uri(MLFLOW_URI); mlflow.set_experiment(EXPERIMENT_NAME)

<Experiment: artifact_location=('file:///D:/ZE5 PORTOFOLIO DS/Q-Factor Prediction in Optical Communication '
 'Systems/mlruns/622530031528693603'), creation_time=1777606059837, experiment_id='622530031528693603', last_update_time=1777606059837, lifecycle_stage='active', name='qfactor_prediction', tags={}, trace_location=None, workspace='default'>

In [2]:
X_train, y_train = load_processed('train')
X_val, y_val = load_processed('val')
scaler = load_scaler()
osnr_idx = FEATURE_COLS.index('OSNR')
# Recover normalized OSNR (in [0,1]) from scaled X
mean_, scale_ = scaler.mean_[osnr_idx], scaler.scale_[osnr_idx]
osnr_train_norm = X_train[:, osnr_idx] * scale_ + mean_
osnr_val_norm = X_val[:, osnr_idx] * scale_ + mean_
device = 'cuda' if torch.cuda.is_available() else 'cpu'; print('device:', device)

device: cpu


In [3]:
bs = 8192
ds = TensorDataset(torch.from_numpy(X_train.astype('float32')), torch.from_numpy(y_train.astype('float32')), torch.from_numpy(osnr_train_norm.astype('float32')))
loader = DataLoader(ds, batch_size=bs, shuffle=True)
model = PINNRegressor(X_train.shape[1]).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
lambda_phys = 0.05; epochs = 15
with mlflow.start_run(run_name='PINN'):
    mlflow.log_param('lambda_phys', lambda_phys)
    for ep in range(epochs):
        model.train(); losses = []
        for xb, yb, ob in loader:
            xb=xb.to(device); yb=yb.to(device); ob=ob.to(device)
            opt.zero_grad()
            preds = model(xb)
            total, dl, pl = pinn_loss(preds, yb, ob, lambda_phys)
            total.backward(); opt.step()
            losses.append(total.item())
        print(f'ep {ep+1:02d}  total={np.mean(losses):.4f}')
    model.eval()
    with torch.no_grad():
        preds = model(torch.from_numpy(X_val.astype('float32')).to(device)).cpu().numpy()
    m = regression_metrics(y_val, preds); print(m); mlflow.log_metrics(m)
torch.save(model.state_dict(), MODELS_DIR / 'pinn.pt')

ep 01  total=6.7921
ep 02  total=0.3709
ep 03  total=0.3032
ep 04  total=0.2695
ep 05  total=0.2434
ep 06  total=0.2257
ep 07  total=0.2136
ep 08  total=0.2054
ep 09  total=0.2003
ep 10  total=0.1973
ep 11  total=0.1942
ep 12  total=0.1915
ep 13  total=0.1892
ep 14  total=0.1876
ep 15  total=0.1860
{'RMSE': 0.13057114757297508, 'MAE': 0.10361731797456741, 'R2': 0.9874625205993652, 'MAPE': 2.0305817127227783}
